# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Tomas Schmieder

**ID**: tas274

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/code/FALL 2025/BEE 4750/hw5-tas274hw5`
   Installed PlotUtils ────────── v1.4.4
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed Measures ─────────── v0.3.3
   Installed GR_jll ───────────── v0.73.18+0
   Installed OpenSSL ──────────── v1.6.0
   Installed MutableArithmetics ─ v1.6.7
   Installed FFMPEG ───────────── v0.4.5
   Installed Pango_jll ────────── v1.57.0+0
   Installed StaticArraysCore ─── v1.4.4
   Installed JSON ─────────────── v1.3.0
   Installed DataStructures ───── v0.19.3
   Installed METIS_jll ────────── v5.1.3+0
   Installed StableRNGs ───────── v1.0.4
   Installed HiGHS ────────────── v1.20.1
   Installed StatsBase ────────── v0.34.8
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed StructUtils ──────── v2.6.0
   Installed ForwardDiff ──────── v1.3.0
   Installed JuMP ─────────────── v1.29.3
   Installed GR ───────────────── v0.73.18
Precompiling project...
    495.0 ms  ✓ Measures
 

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

#### Problem 1.1 Writeup

The overall recycling fraction was found to be 0.378 and the overall ash fractiion was found to be 0.164 for the waste produced by each city. This was done by simply summing the multiplication of their respective ash or recycling rate with the mass fraction.

In [20]:
# Problem 1.1 Code
using DataFrames
df = DataFrame(
    Component = [
        "Food Wastes", "Paper & Cardboard", "Plastics", "Textiles",
        "Rubber, Leather", "Wood", "Yard Wastes", "Glass",
        "Ferrous", "Aluminum", "Other Metal", "Miscellaneous"
    ],
    Mass_Fraction = [0.15, 0.40, 0.05, 0.03, 0.02, 0.05, 0.18, 0.04, 0.02, 0.02, 0.01, 0.03],
    Combustion_Ash_Rate = [0.08, 0.07, 0.05, 0.10, 0.15, 0.02, 0.02, 1.00, 1.00, 1.00, 1.00, 0.70],
    MRF_Recycling_Rate = [0.00, 0.55, 0.15, 0.10, 0.00, 0.30, 0.40, 0.60, 0.75, 0.80, 0.50, 0.00]
)
df.RecyclingContribution = df.Mass_Fraction .* df.MRF_Recycling_Rate
R_overall = sum(df.RecyclingContribution)
df.AshContribution = df.Mass_Fraction .* data.Combustion_Ash_Rate
A_WTE = sum(df.AshContribution)

println("R_overall: ", round(R_overall; digits=3))
println("A_WTE: ", round(A_WTE; digits=4))


R_overall: 0.378
A_WTE: 0.1641


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

#### Problem 1.2 Writeup
The decision variables are as follows, 
- The Implementation Decision $y_f$ which is binary, $\in\{0,1\}$. 
- The Original MSF Flow $x_{c,f}$ which is greater then or equal to zero. It is the mass of the municipal solid waste transporte daily from city $c\in\{1,2,3\}$ to facilitiy $f\in\{LF, MRF, WTE\}$. The units are mg/day
- The Residual MSF Flow $z_{redidual,f}$ which is greater then or equal to zero. It is the mass of non-recycled resiudals transported daily from the MRF to disposal facility, here f is only in LF and WTE. The units are mg/day
- The Ash from FSW $a_{msw}$ which is greater then or equal to zero. It is the mass of ash produced daily at the WTE facilitiy from burning MSW and sent to LF. Units are mg/day
- The Ash from Residuals $a_{residual}$ which is greater then or equal to zero. It is the mass of ash produced daily at the WTE facility from burning MRF residuals and sent to LF. mg/day

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

#### Problem 1.3 Writeup

The components of the objective function function are as follows, first the fixed costs, we can define this as,
$$C_{fixed}=\sum_f~(Fixed~Cost_f~\cdot~y_f)$$
Where $f\in\{LF, MRF, WTE\}$ and costs for each of them are 2000, 1500, 2500 $/day respectively.

The second component is the tipping cost. This is the cost per Mg of waste recieved at each of the faciltiies,
$$C_{tip}=TC_{LF}\cdot F_{LF} + TC_{MRF}\cdot F_{MRF} + TC_{WTE}\cdot F_{WTE}$$
Where, for each faciltiy we have $F$ as,
$$F_{LF}=\sum_c x_{c,LF} + z_{residual, LF} + a_{msw} + a_{residual}$$
$$F_{MRF}=\sum_c x_{c,MRF} $$
$$F_{WTE}=\sum_c x_{c,WTE} + z_{resisdual,WTE}$$

The third component is the recycling costs,
$$C_{recycling}=RC_{MRF}\cdot{F_{recycled}}$$
Where $RC=40 $/mg$ and, the total mass recycled is,
$$F_{recycled}=R_{MRF}\cdot{F_{MRF}}=0.40\cdot\sum_c x_{c,MRF}$$


The fourth component is the transportation costs,
$$C_{transportation}=1.5~~dollar/mg\cdot km \sum (Flow\cdot Distance)$$
Where each of our flows are, the city to facility flows, $\sum_c \sum_f x_{c,f} \cdot D_{c,f} $, the MRF Residuals to the disposal flows, $z_{residual,LF}\cdot D_{MRF, LF} + z_{residual, WTE} \cdot D_{MRF, WTE} $, and the WTE Ash to LF flow, $(a_{MSW} + a_{Residual})\cdot D_{WTE, LF}$

So now in total our final objective function can be listed as,
$$
\begin{align*}
\text{Minimize} Z &= \sum_{f} \left( \text{Fixed Cost}_f \times y_f \right) \\
&\quad + \left[ \text{Tipping Cost}_{\text{LF}} \left( \sum_{c} x_{c, \text{LF}} + z_{\text{res, LF}} + a_{msw} + a_{res} \right) \right] \\
&\quad + \left[ \text{Tipping Cost}_{\text{MRF}} \left( \sum_{c} x_{c, \text{MRF}} \right) \right] \\
&\quad + \left[ \text{Tipping Cost}_{\text{WTE}} \left( \sum_{c} x_{c, \text{WTE}} + z_{\text{res, WTE}} \right) \right] \\
&\quad + \left[ \text{Recycling Cost}_{\text{MRF}} \times 0.40 \left( \sum_{c} x_{c, \text{MRF}} \right) \right] \\
&\quad + 1.5 \left[ \sum_{c}\sum_{f} x_{c,f} D_{c,f} + z_{\text{res, LF}} D_{\text{MRF, LF}} + z_{\text{res, WTE}} D_{\text{MRF, WTE}} \right. \\
&\qquad \left. + (a_{msw} + a_{res}) D_{\text{WTE, LF}} \right]
\end{align*}
$$


#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### Problem 1.4 Writeup

The constraints are as follows,

1. The cities sum of MSW flows that are outputs must equal its total daily production/demand
$$\sum_f x_{c,f}=Demand_c$$
2. The total mass flow that enters a facility cant exceed its maximu daily capacity.
$$\sum_c x_{c,LF} = z_{residual,LF} + a_{MSW} + a_{residual} \leq Capacity_{LF} $$
$$\sum_c x_{c,MRF} \leq Capacity_{MRF}$$
$$\sum_c x_{c,WTE} + z_{resiudal, WTE} \leq Capacity_{WTE}$$
3. A constraint ensuring that if any waste is sent to a facility that facility must be used, this is done by using the total flow
$$TF_f \leq Capacity_f \cdot y_f $$

$$\sum_c x_{c,LF} + z_{residua, LF} + a_{MSW} + a_{residual} \leq 200 \cdot y_{LF} $$
$$\sum_c x_{c,MRF} \leq 350\cdot y_{MRF}$$
$$\sum_c x_{c,WTE} + z_{residual,WTE} \leq 210 \cdot y_{WTE}$$
4. The mass balance constraint ensuring that the output mass flows from the MRF and WTE are calculated based ont their input flows. To do this we need constraints for the MRF output mass balance, and then one for the WTE ash production mass balance.
$$z_{residual,LF}+z_{residual,WTE}=(1-R_{MRF})\sum_c x_{c,MRF}$$
$$a_{MSW}=0.1641\cdot \sum_c x_{c,WTE}$$
$$a_{residual} = 0.16 \cdot z_{residual, WTE}$$
5. Finally the last constraints are just non negative ones,
$$x_{c,f}\leq 0,~~~\forall c,f$$
$$z_{residual,f}\leq 0,~~~f\in\{LF, WTE\}$$
$$a_{MSW}\leq0$$
$$a_{residual}\leq0$$
$$y_f\in\{0,1\},~~~f\in\{LF,MRF,WTE\}$$

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [ ]:
Cities = ["1", "2", "3"]
Facilities = ["LF", "MRF", "WTE"]

Capacity = Dict("LF" => 200, "MRF" => 350, "WTE" => 210)
FixedCost = Dict("LF" => 2000, "MRF" => 1500, "WTE" => 2500)
TippingCost = Dict("LF" => 50, "MRF" => 7, "WTE" => 60)
RecyclingCostMRF = 40
CityDemand = Dict("1" => 100, "2" => 90, "3" => 120)
TransportCostRate = 1.5
Distance = Dict(
    ("City1", "LF") => 5, ("City1", "MRF") => 30, ("City1", "WTE") => 15,
    ("City2", "LF") => 15, ("City2", "MRF") => 25, ("City2", "WTE") => 10,
    ("City3", "LF") => 13, ("City3", "MRF") => 45, ("City3", "WTE") => 20,
    ("MRF", "LF") => 32,
    ("MRF", "WTE") => 15,
    ("WTE", "LF") => 18,
    ("WTE", "MRF") => 15
)
R_MRF = 0.40
A_WTE_MSW = 0.1641
A_ASH_RES = 0.16
const M= sum(values(CityDemand))

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, y[f in Facilities], Bin)
@variable(model, x[c in Cities, f in Facilities] >= 0)
@variable(model, z_res[f in ["LF", "WTE"]] >= 0)
@variable(model, a_msw_lf >= 0)
@variable(model, a_res_lf >= 0)

const F_MRF = sum(x[c, "MRF"] for c in Cities)
const F_Recycled = R_MRF * F_MRF
const F_WTE_ASH_OUT = a_msw_lf + a_res_lf
const F_WTE_IN = sum(x[c, "WTE"] for c in Cities) + z_res["WTE"]
const F_LF_WASTE_IN = sum(x[c, "LF"] for c in Cities) + z_res["LF"]

@objective(model, Min,
    sum(FixedCost[f] * y[f] for f in Facilities) + # Fixed Costs
    (TippingCost["LF"] * (F_LF_WASTE_IN + F_WTE_ASH_OUT)) +
    (TippingCost["MRF"] * F_MRF) +
    (TippingCost["WTE"] * F_WTE_IN) +
    (RECYCLING_COST_MRF * F_Recycled) + # Recycling Cost
    TransportCostRate * ( # Transportation Costs
        sum(x[c, f] * Distance[(c, f)] for c in Cities for f in Facilities) +
        z_res["LF"] * Distance[("MRF", "LF")] +
        z_res["WTE"] * Distance[("MRF", "WTE")] +
        F_WTE_ASH_OUT * Distance[("WTE", "LF")]
    )
)

@constraint(model, [c in Cities], sum(x[c,f] for f in Facilities) == CityDemand[c])

@constraint(model, LF_Capacity, (F_LF_WASTE_IN + F_WTE_ASH_OUT) <= Capacity["LF"] * y["LF"])
@constraint(model, MRF_Capacity, F_MRF <= Capacity["MRF"] * y["MRF"])
@constraint(model, WTE_Capacity, F_WTE_IN <= Capacity["WTE"] * y["WTE"])

# C. MRF Mass Balance
@constraint(model, MRF_Residuals_Balance, sum(z_res[f] for f in ["LF", "WTE"]) == (1 - R_MRF) * F_MRF)

# D. WTE Ash Production
@constraint(model, Ash_MSW, a_msw_lf == A_WTE_MSW * sum(x[c, "WTE"] for c in Cities))
@constraint(model, Ash_Res, a_res_lf == A_ASH_RES * z_res["WTE"])

# --- 5. Solve ---
optimize!(model)

InterruptException: InterruptException:

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is 1500 MW. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.